# DepthForge: train, route & evaluate on NYU Depth V2 (Colab / Kaggle GPU)
Runtime -> Change runtime type -> **T4 GPU**. Kaggle (4 CPU cores) is better than Colab (2 cores): the training data pipeline runs the classical filters on the CPU and is usually the bottleneck, not the GPU.

Order: 1) train longer, 2) fit the learned router, 3) evaluate on the unseen test split, 4) export ONNX.
Replace the repo URL with yours.

In [ ]:
!git clone https://github.com/<your-user>/depthforge.git
%cd depthforge
!pip -q install opencv-python-headless scikit-image h5py gradio

In [ ]:
# NYU v2 labeled set (2.8 GB, 1449 RGB-D pairs). Standard split: 0-999 train(900)/val(100), 1000-1448 test
!wget -q --show-progress -c http://horatio.cs.nyu.edu/mit/silberman/nyu_depth_v2/nyu_depth_v2_labeled.mat
# mirror if the NYU server is slow:
# !pip -q install gdown; # or use the Kaggle dataset 'nyu-depth-v2-labeled'

In [ ]:
import torch; print(torch.cuda.get_device_name(0))
# Continue from the current checkpoint for a long run. AMP = fp16 on the GPU, EMA = averaged weights (usually +1-3 % RMSE).
# Back up the old weights first so you can compare / roll back.
!cp weights/depthforge_net.pt weights/depthforge_net_prev.pt
!python scripts/train.py --data nyu --mat nyu_depth_v2_labeled.mat --minutes 150 --crop 128 --batch 32 --workers 4 --amp --ema 0.999 --lr 1e-3 --init weights/depthforge_net.pt
# Bigger net from scratch (needs more time, cannot warm-start from the width-32 checkpoint):
# !python scripts/train.py --data nyu --mat nyu_depth_v2_labeled.mat --minutes 240 --crop 128 --batch 32 --workers 4 --amp --width 48 --out weights/depthforge_net_w48.pt


**Learned router.** Fits on NYU *validation* images 900-999 only (never test) with 5-fold CV grouped by image. If the CV report says it is not better than the hand-set rules, delete `weights/router.json` and the app falls back to the rules automatically.

In [ ]:
!python scripts/train_router.py --data nyu --mat nyu_depth_v2_labeled.mat --n 100
!cat results/router_report.md


In [ ]:
# Submission evaluation on the unseen NYU test images (reduce --n for a quick run)
!python scripts/evaluate.py --data nyu --mat nyu_depth_v2_labeled.mat --n 100 --team TigerFC --usns '1RUA24CSE0021,1RUA24CSE0401,1RUA24CSE0427,1RUA24CSE0490'
!python scripts/quantize.py --n 10


In [ ]:
from google.colab import files
!python scripts/export_onnx.py
for f in ['weights/depthforge_net.pt','weights/depthforge_net.onnx','weights/router.json']:
    files.download(f)
!zip -qr results.zip results && echo ok
files.download('results.zip')


Copy `weights/depthforge_net.pt`, `weights/depthforge_net.onnx` and `weights/router.json` into the repo's `weights/` folder, re-run `scripts/evaluate.py` locally or here, and redeploy.